# Chapter 40: Visual Inertial Fusion

<a href="../lite/lab/index.html?path=ch40_visual_inertial.ipynb" target="_blank" style="display:inline-block;padding:8px 18px;background:#1976d2;color:white;border-radius:5px;text-decoration:none;font-weight:bold;font-size:0.95em;">&#9654; Open in JupyterLite: run and edit this notebook</a>

*Runs entirely in your browser; no installation required.*

**How to use:** Edit the parameter values in each cell and re-run it to explore.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

%matplotlib inline
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

The camera and the IMU are perfect partners. The camera is slow but drift free. The IMU is fast but drifts wildly. Together, they cover each other's weaknesses. The camera anchors the IMU to prevent drift. The IMU fills in between camera frames for smooth, high rate tracking. This marriage is called **Visual Inertial Odometry** (VIO), and it runs on every modern smartphone.

This chapter builds a complete VIO simulation, starting from the complementary nature of the two sensors and culminating in a fused estimator that outperforms either sensor alone.

```{admonition} What you will build
:class: tip

- Fuse camera (10 Hz) and IMU (200 Hz) into a single high rate, drift free trajectory
- Implement loose coupling: run visual and inertial estimators separately, then fuse with a Kalman filter
- Show that the fused estimate is dramatically better than either sensor alone
- Understand the difference between tight and loose coupling architectures

**Real world application:** Visual Inertial Odometry (VIO) runs on every modern smartphone for AR. After this chapter, you will understand the sensor fusion that makes ARKit, ARCore, and drone autopilots work.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **VINS-Mono / VINS-Fusion** | State of the art VIO with IMU preintegration |
| **ORB-SLAM3 (inertial mode)** | Visual-inertial SLAM with IMU initialization |
| **MSCKF** | Multi-State Constraint Kalman Filter for efficient VIO |
| **Basalt** | Visual-inertial odometry with non-linear optimization |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

---
## 40.1 Complementarity: Why Camera + IMU?

| Property | Camera | IMU |
|----------|--------|-----|
| **Rate** | 10 to 30 Hz | 200 to 1000 Hz |
| **Drift** | None (absolute features) | Unbounded (bias accumulates) |
| **Latency** | High (image processing) | Low (direct readout) |
| **Failure modes** | Blur, occlusion, darkness | None (works always) |
| **Scale** | Unknown (monocular) | Known (accelerometer reads $g$) |

The IMU provides:
- **High rate** predictions between camera frames
- **Scale** from gravity measurement
- **Roll/pitch** from gravity direction
- **Robustness** when visual tracking fails

The camera provides:
- **Drift free** position anchoring
- **Bias observability** (camera corrections let us estimate IMU biases)
- **Loop closure** for global consistency

In [ ]:
def generate_trajectory(duration, dt_fine, shape='loop'):
    """Generate a smooth 2D ground truth trajectory.
    
    Returns: t, x, y, vx, vy, ax, ay, theta, omega
    """
    N = int(duration / dt_fine)
    t = np.arange(N) * dt_fine
    freq = 2 * np.pi / duration
    
    if shape == 'loop':
        x = 4.0 * np.sin(freq * t)
        y = 2.0 * np.sin(2 * freq * t)
    elif shape == 'circle':
        x = 3.0 * np.cos(freq * t) - 3.0
        y = 3.0 * np.sin(freq * t)
    else:
        x = 2.0 * t / duration * np.cos(0.3 * t)
        y = 2.0 * t / duration * np.sin(0.3 * t)
    
    vx = np.gradient(x, dt_fine)
    vy = np.gradient(y, dt_fine)
    ax = np.gradient(vx, dt_fine)
    ay = np.gradient(vy, dt_fine)
    theta = np.unwrap(np.arctan2(vy, vx))
    omega = np.gradient(theta, dt_fine)
    
    return t, x, y, vx, vy, ax, ay, theta, omega


def simulate_imu(ax, ay, theta, omega, dt, a_noise, g_noise, a_bias, g_bias):
    """Simulate noisy body-frame IMU measurements from world-frame truth."""
    N = len(ax)
    # Transform world acceleration to body frame
    ax_body = np.zeros(N)
    ay_body = np.zeros(N)
    for i in range(N):
        c, s = np.cos(theta[i]), np.sin(theta[i])
        ax_body[i] =  c * ax[i] + s * ay[i]
        ay_body[i] = -s * ax[i] + c * ay[i]
    
    # Add noise and bias
    meas_ax = ax_body + a_bias[0] + np.random.normal(0, a_noise, N)
    meas_ay = ay_body + a_bias[1] + np.random.normal(0, a_noise, N)
    meas_w  = omega   + g_bias    + np.random.normal(0, g_noise, N)
    
    return meas_ax, meas_ay, meas_w


def integrate_imu_2d(ax_b, ay_b, omega, dt, x0, y0, vx0, vy0, theta0):
    """Full 2D IMU integration: gyro to heading, accel to position."""
    N = len(ax_b)
    theta = np.zeros(N); theta[0] = theta0
    vx = np.zeros(N); vy = np.zeros(N); vx[0] = vx0; vy[0] = vy0
    x = np.zeros(N); y = np.zeros(N); x[0] = x0; y[0] = y0
    for i in range(1, N):
        theta[i] = theta[i-1] + omega[i-1] * dt
        c, s = np.cos(theta[i-1]), np.sin(theta[i-1])
        aw_x = c * ax_b[i-1] - s * ay_b[i-1]
        aw_y = s * ax_b[i-1] + c * ay_b[i-1]
        vx[i] = vx[i-1] + aw_x * dt
        vy[i] = vy[i-1] + aw_y * dt
        x[i] = x[i-1] + vx[i-1] * dt
        y[i] = y[i-1] + vy[i-1] * dt
    return x, y, vx, vy, theta


def simulate_camera(x_true, y_true, cam_noise, cam_rate, imu_rate):
    """Simulate camera position observations at a lower rate.
    
    Returns: cam_indices (indices into the IMU-rate arrays), cam_x, cam_y
    """
    ratio = int(imu_rate / cam_rate)
    cam_indices = np.arange(0, len(x_true), ratio)
    cam_x = x_true[cam_indices] + np.random.normal(0, cam_noise, len(cam_indices))
    cam_y = y_true[cam_indices] + np.random.normal(0, cam_noise, len(cam_indices))
    return cam_indices, cam_x, cam_y

print('Helper functions defined.')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
duration      = 30.0      # trajectory duration (s)
imu_rate      = 200       # IMU rate (Hz)        try 100, 200, 500
camera_rate   = 10        # camera rate (Hz)     try 5, 10, 30
cam_noise     = 0.05      # camera position noise σ (m)   try 0.01, 0.1
imu_a_noise   = 0.04      # accelerometer noise σ (m/s²)
imu_g_noise   = 0.003     # gyroscope noise σ (rad/s)
imu_a_bias    = np.array([0.015, -0.008])   # accel bias (m/s²)
imu_g_bias    = 0.0005    # gyro bias (rad/s)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
dt_imu = 1.0 / imu_rate

# Ground truth
t, gt_x, gt_y, gt_vx, gt_vy, gt_ax, gt_ay, gt_th, gt_w = \
    generate_trajectory(duration, dt_imu, shape='loop')

# Simulate IMU
imu_ax, imu_ay, imu_w = simulate_imu(
    gt_ax, gt_ay, gt_th, gt_w, dt_imu,
    imu_a_noise, imu_g_noise, imu_a_bias, imu_g_bias)

# Simulate camera
cam_idx, cam_x, cam_y = simulate_camera(
    gt_x, gt_y, cam_noise, camera_rate, imu_rate)

# IMU-only integration
imu_x, imu_y, _, _, _ = integrate_imu_2d(
    imu_ax, imu_ay, imu_w, dt_imu,
    gt_x[0], gt_y[0], gt_vx[0], gt_vy[0], gt_th[0])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(gt_x, gt_y, 'k', lw=2.5, label='Ground truth', zorder=5)
axes[0].plot(imu_x, imu_y, 'tomato', lw=1.2, alpha=0.7, label='IMU only (drifts)')
axes[0].scatter(cam_x, cam_y, s=8, color='steelblue', alpha=0.5, label=f'Camera ({camera_rate} Hz)', zorder=3)
axes[0].plot(gt_x[0], gt_y[0], 'o', color='forestgreen', ms=10, zorder=6)
axes[0].set_xlabel('x (m)'); axes[0].set_ylabel('y (m)')
axes[0].set_title('Three Information Sources'); axes[0].legend(fontsize=9)
axes[0].set_aspect('equal')

# Error comparison
imu_err = np.sqrt((imu_x - gt_x)**2 + (imu_y - gt_y)**2)
# Camera error at camera times
cam_err = np.sqrt((cam_x - gt_x[cam_idx])**2 + (cam_y - gt_y[cam_idx])**2)
t_cam = t[cam_idx]

axes[1].plot(t, imu_err, 'tomato', lw=1.5, label='IMU only error')
axes[1].scatter(t_cam, cam_err, s=10, color='steelblue', alpha=0.5, label='Camera error (at cam frames)')
axes[1].axhline(np.mean(cam_err), color='steelblue', ls='--', alpha=0.5, label=f'Camera mean err = {np.mean(cam_err):.3f} m')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Position error (m)')
axes[1].set_title('Error Over Time'); axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

print(f'IMU final error:       {imu_err[-1]:.2f} m (grows without bound)')
print(f'Camera mean error:     {np.mean(cam_err):.3f} m (bounded, but only {camera_rate} Hz)')

**The problem is clear:** The IMU gives smooth, high rate tracking that drifts. The camera gives drift free positions but only at low rate. We need to *fuse* them.

---
## 40.2 Loose Coupling with a Kalman Filter

The simplest fusion approach: **loose coupling**.

1. Between camera frames, use IMU integration for **prediction** (high rate).
2. At each camera frame, **correct** the predicted state with the camera observation.

This is exactly the Kalman filter from Chapter 16, with the IMU as the motion model and the camera as the measurement model.

**State vector:** $\mathbf{x} = [x, y, v_x, v_y, \theta]$

**Prediction** (using IMU at each IMU timestep):
$$\hat{\theta}_{k+1} = \hat{\theta}_k + \omega_k \Delta t$$
$$\hat{v}_{k+1} = \hat{v}_k + R(\hat{\theta}_k) \cdot \mathbf{a}^b_k \cdot \Delta t$$
$$\hat{p}_{k+1} = \hat{p}_k + \hat{v}_k \cdot \Delta t$$

**Update** (at camera frame):
$$\mathbf{z} = [x_{\text{cam}}, y_{\text{cam}}]$$
$$K = P H^\top (H P H^\top + R_{\text{cam}})^{-1}$$
$$\hat{\mathbf{x}} \leftarrow \hat{\mathbf{x}} + K(\mathbf{z} - H\hat{\mathbf{x}})$$

In [ ]:
class LooseCoupledVIO:
    """Loosely coupled Visual Inertial Odometry using an EKF.
    
    State: [x, y, vx, vy, theta] (5 dimensional)
    IMU provides: body-frame acceleration and angular velocity (prediction)
    Camera provides: position observations (update)
    """
    def __init__(self, x0, y0, vx0, vy0, theta0, 
                 imu_accel_noise, imu_gyro_noise, cam_pos_noise):
        self.state = np.array([x0, y0, vx0, vy0, theta0])
        self.P = np.diag([0.01, 0.01, 0.1, 0.1, 0.01])  # initial covariance
        
        # Process noise from IMU
        self.Q_accel = imu_accel_noise**2
        self.Q_gyro  = imu_gyro_noise**2
        
        # Measurement noise from camera
        self.R_cam = cam_pos_noise**2 * np.eye(2)
        
        # Observation matrix: we observe [x, y]
        self.H = np.zeros((2, 5))
        self.H[0, 0] = 1.0  # observe x
        self.H[1, 1] = 1.0  # observe y
    
    def predict(self, ax_body, ay_body, omega, dt):
        """IMU prediction step."""
        x, y, vx, vy, theta = self.state
        
        # Rotate body acceleration to world frame
        c, s = np.cos(theta), np.sin(theta)
        ax_w = c * ax_body - s * ay_body
        ay_w = s * ax_body + c * ay_body
        
        # State prediction
        new_x = x + vx * dt
        new_y = y + vy * dt
        new_vx = vx + ax_w * dt
        new_vy = vy + ay_w * dt
        new_theta = theta + omega * dt
        
        self.state = np.array([new_x, new_y, new_vx, new_vy, new_theta])
        
        # Jacobian of the state transition
        F = np.eye(5)
        F[0, 2] = dt  # dx/dvx
        F[1, 3] = dt  # dy/dvy
        F[2, 4] = (-s * ax_body - c * ay_body) * dt  # dvx/dtheta
        F[3, 4] = ( c * ax_body - s * ay_body) * dt  # dvy/dtheta
        
        # Process noise covariance
        Q = np.diag([
            self.Q_accel * dt**2 * 0.25,  # position noise from accel
            self.Q_accel * dt**2 * 0.25,
            self.Q_accel * dt,             # velocity noise from accel
            self.Q_accel * dt,
            self.Q_gyro * dt               # heading noise from gyro
        ])
        
        self.P = F @ self.P @ F.T + Q
    
    def update(self, z_cam):
        """Camera measurement update."""
        innovation = z_cam - self.H @ self.state
        S = self.H @ self.P @ self.H.T + self.R_cam
        K = self.P @ self.H.T @ np.linalg.inv(S)
        
        self.state = self.state + K @ innovation
        self.P = (np.eye(5) - K @ self.H) @ self.P
        
        return innovation  # for diagnostics
    
    @property
    def position(self):
        return self.state[:2]
    
    @property
    def velocity(self):
        return self.state[2:4]
    
    @property
    def heading(self):
        return self.state[4]
    
    @property
    def position_cov(self):
        return self.P[:2, :2]

print('LooseCoupledVIO class defined.')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
# Reuse trajectory from Section 40.1, or regenerate
fuse_duration   = 30.0     # seconds
fuse_imu_rate   = 200      # Hz
fuse_cam_rate   = 10       # Hz        try 5, 10, 30
fuse_cam_noise  = 0.05     # m         try 0.02, 0.1, 0.3
fuse_a_noise    = 0.04     # m/s²
fuse_g_noise    = 0.003    # rad/s
fuse_a_bias     = np.array([0.015, -0.008])  # m/s²
fuse_g_bias     = 0.0005   # rad/s
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
dt_f = 1.0 / fuse_imu_rate
cam_every = int(fuse_imu_rate / fuse_cam_rate)

# Ground truth
t_f, gx, gy, gvx, gvy, gax, gay, gth, gw = \
    generate_trajectory(fuse_duration, dt_f, shape='loop')
N_f = len(t_f)

# Simulate sensors
f_imu_ax, f_imu_ay, f_imu_w = simulate_imu(
    gax, gay, gth, gw, dt_f, fuse_a_noise, fuse_g_noise, fuse_a_bias, fuse_g_bias)
f_cam_idx, f_cam_x, f_cam_y = simulate_camera(
    gx, gy, fuse_cam_noise, fuse_cam_rate, fuse_imu_rate)

# === Run three estimators ===

# 1. IMU only
imu_only_x, imu_only_y, _, _, _ = integrate_imu_2d(
    f_imu_ax, f_imu_ay, f_imu_w, dt_f,
    gx[0], gy[0], gvx[0], gvy[0], gth[0])

# 2. Camera only: interpolate between camera observations
cam_only_x = np.interp(np.arange(N_f), f_cam_idx, f_cam_x)
cam_only_y = np.interp(np.arange(N_f), f_cam_idx, f_cam_y)

# 3. Fused VIO
vio = LooseCoupledVIO(
    gx[0], gy[0], gvx[0], gvy[0], gth[0],
    fuse_a_noise, fuse_g_noise, fuse_cam_noise)

fused_x = np.zeros(N_f)
fused_y = np.zeros(N_f)
fused_x[0] = gx[0]; fused_y[0] = gy[0]

cam_counter = 0
cam_set = set(f_cam_idx)

for i in range(1, N_f):
    # IMU prediction at every step
    vio.predict(f_imu_ax[i-1], f_imu_ay[i-1], f_imu_w[i-1], dt_f)
    
    # Camera update at camera rate
    if i in cam_set:
        ci = np.searchsorted(f_cam_idx, i)
        if ci < len(f_cam_x):
            vio.update(np.array([f_cam_x[ci], f_cam_y[ci]]))
    
    fused_x[i] = vio.position[0]
    fused_y[i] = vio.position[1]

# Compute errors
err_imu  = np.sqrt((imu_only_x - gx)**2 + (imu_only_y - gy)**2)
err_cam  = np.sqrt((cam_only_x - gx)**2 + (cam_only_y - gy)**2)
err_fuse = np.sqrt((fused_x - gx)**2 + (fused_y - gy)**2)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].plot(gx, gy, 'k', lw=3, label='Ground truth', zorder=5)
axes[0].plot(imu_only_x, imu_only_y, 'tomato', lw=1, alpha=0.7, label='IMU only')
axes[0].plot(cam_only_x, cam_only_y, 'orange', lw=1, alpha=0.7, label='Camera only (interp)')
axes[0].plot(fused_x, fused_y, 'steelblue', lw=2, label='Fused VIO', zorder=4)
axes[0].plot(gx[0], gy[0], 'o', color='forestgreen', ms=10, zorder=6)
axes[0].set_xlabel('x (m)'); axes[0].set_ylabel('y (m)')
axes[0].set_title('Trajectory Comparison: Three Estimators')
axes[0].legend(fontsize=9); axes[0].set_aspect('equal')

axes[1].plot(t_f, err_imu, 'tomato', lw=1, alpha=0.7, label='IMU only')
axes[1].plot(t_f, err_cam, 'orange', lw=1, alpha=0.7, label='Camera only')
axes[1].plot(t_f, err_fuse, 'steelblue', lw=1.5, label='Fused VIO')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Position error (m)')
axes[1].set_title('Position Error Over Time'); axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()

print(f'Mean position error:')
print(f'  IMU only:     {np.mean(err_imu):.3f} m')
print(f'  Camera only:  {np.mean(err_cam):.3f} m')
print(f'  Fused VIO:    {np.mean(err_fuse):.3f} m')
print(f'\nFusion improvement over IMU:    {np.mean(err_imu)/np.mean(err_fuse):.1f}x')
print(f'Fusion improvement over Camera: {np.mean(err_cam)/np.mean(err_fuse):.1f}x')

**What to observe:** The fused trajectory (steelblue) closely follows the ground truth. It is smooth (thanks to the IMU) and drift free (thanks to the camera). The IMU alone drifts badly. The camera alone is noisy and only available at low rate. The fusion is strictly better than either input.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
# Show the effect of camera rate on fusion quality
test_cam_rates = [2, 5, 10, 20, 30]   # Hz
rate_cam_noise = 0.05
rate_duration  = 30.0
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
dt_rate = 1.0 / fuse_imu_rate
t_r, rx, ry, rvx, rvy, rax, ray, rth, rw = \
    generate_trajectory(rate_duration, dt_rate, shape='loop')
N_r = len(t_r)

r_imu_ax, r_imu_ay, r_imu_w = simulate_imu(
    rax, ray, rth, rw, dt_rate, fuse_a_noise, fuse_g_noise, fuse_a_bias, fuse_g_bias)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_rate = ['tomato', 'orange', 'steelblue', 'forestgreen', 'purple']

mean_errors = []
for j, cr in enumerate(test_cam_rates):
    np.random.seed(42 + j)
    ci, cx, cy = simulate_camera(rx, ry, rate_cam_noise, cr, fuse_imu_rate)
    
    vio_r = LooseCoupledVIO(
        rx[0], ry[0], rvx[0], rvy[0], rth[0],
        fuse_a_noise, fuse_g_noise, rate_cam_noise)
    
    fx = np.zeros(N_r); fy = np.zeros(N_r)
    fx[0] = rx[0]; fy[0] = ry[0]
    cs = set(ci)
    
    for i in range(1, N_r):
        vio_r.predict(r_imu_ax[i-1], r_imu_ay[i-1], r_imu_w[i-1], dt_rate)
        if i in cs:
            idx = np.searchsorted(ci, i)
            if idx < len(cx):
                vio_r.update(np.array([cx[idx], cy[idx]]))
        fx[i] = vio_r.position[0]
        fy[i] = vio_r.position[1]
    
    err = np.sqrt((fx - rx)**2 + (fy - ry)**2)
    mean_errors.append(np.mean(err))
    axes[0].plot(t_r, err, color=colors_rate[j], lw=1, alpha=0.8, label=f'{cr} Hz')

axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Position error (m)')
axes[0].set_title('Fusion Error vs Camera Rate'); axes[0].legend(fontsize=9)

axes[1].plot(test_cam_rates, mean_errors, 'o-', color='steelblue', lw=2, ms=8)
axes[1].set_xlabel('Camera rate (Hz)'); axes[1].set_ylabel('Mean position error (m)')
axes[1].set_title('Mean Error vs Camera Rate')

plt.tight_layout(); plt.show()

for cr, me in zip(test_cam_rates, mean_errors):
    print(f'  Camera {cr:2d} Hz: mean error = {me:.4f} m')

**Key insight:** More frequent camera updates reduce the drift between corrections. Even a modest camera rate (10 Hz) provides substantial improvement over IMU alone. Diminishing returns appear above roughly 20 Hz.

### Tight vs Loose Coupling

The approach above is **loose coupling**: we run visual odometry and IMU integration as independent modules, then fuse their outputs.

**Tight coupling** goes deeper: it optimizes over raw visual features (pixel observations) and raw IMU measurements simultaneously. The advantages:

1. **Better accuracy**: features that are poorly constrained by vision alone can be helped by IMU constraints.
2. **Better bias estimation**: the optimizer can adjust IMU biases to make visual and inertial constraints consistent.
3. **More robust**: even if only 1 or 2 features are visible (not enough for standalone VO), the IMU provides enough constraints.

The factor graph for tight coupling looks like this:

```
Pose[0] ---IMU---> Pose[1] ---IMU---> Pose[2] ---IMU---> Pose[3]
  |                   |                   |                   |
  |  +--Feat[A]-------+                   |                   |
  |  |                |                   |                   |
  +--+     +----------+    +--Feat[B]-----+                   |
     |     |               |              |                   |
     +-----+               +--------------+    +--Feat[C]-----+
                                               |              |
                                               +--------------+
```

Each **IMU factor** constrains the relative motion between consecutive poses. Each **visual factor** constrains the pose relative to a 3D feature. Optimizing all of them jointly is tight coupling.

Modern VIO systems (Apple ARKit, Google ARCore, VINS-Mono) all use tight coupling. In the simulation below, we focus on loose coupling since it demonstrates the core principles clearly.

---
## 40.3 IMU Preintegration (Conceptual)

When we integrate IMU readings into absolute poses, the result depends on the **linearization point** (the current state estimate). If the optimizer updates the state, we would need to re-integrate all the IMU data. This is wasteful.

**Preintegration** solves this by integrating IMU readings into a **relative** measurement between two keyframes, independent of the absolute state:

$$\Delta R_{ij} = \prod_{k=i}^{j-1} \text{Exp}((\omega_k - b_g) \Delta t)$$
$$\Delta v_{ij} = \sum_{k=i}^{j-1} \Delta R_{ik} (a_k - b_a) \Delta t$$
$$\Delta p_{ij} = \sum_{k=i}^{j-1} \left[ \Delta v_{ik} \Delta t + \frac{1}{2} \Delta R_{ik} (a_k - b_a) \Delta t^2 \right]$$

These preintegrated quantities ($\Delta R$, $\Delta v$, $\Delta p$) are computed *once* from the raw IMU data. They can then be used as a **single factor** connecting keyframe $i$ to keyframe $j$ in the factor graph.

When the bias estimate changes, we do not need to re-integrate. Instead, we apply a first order correction:

$$\Delta R_{ij}(b_g) \approx \Delta R_{ij}(\hat{b}_g) \cdot \text{Exp}\left(\frac{\partial \Delta R_{ij}}{\partial b_g} \delta b_g\right)$$

This trick (from the Forster et al. 2017 paper) makes preintegration both efficient and compatible with iterative optimization.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
preint_dt      = 0.005   # IMU timestep
preint_dur     = 10.0    # total duration
keyframe_rate  = 2       # keyframes per second
preint_a_noise = 0.04
preint_g_noise = 0.003
preint_a_bias  = np.array([0.01, -0.005])
preint_g_bias  = 0.0004
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(55)
N_pi = int(preint_dur / preint_dt)
t_pi, px, py, pvx, pvy, pax, pay, pth, pw = \
    generate_trajectory(preint_dur, preint_dt, shape='loop')

pi_imu_ax, pi_imu_ay, pi_imu_w = simulate_imu(
    pax, pay, pth, pw, preint_dt,
    preint_a_noise, preint_g_noise, preint_a_bias, preint_g_bias)

# Preintegrate between keyframes
kf_interval = int(1.0 / (keyframe_rate * preint_dt))  # IMU steps between keyframes
n_keyframes = int(preint_dur * keyframe_rate)

# Store preintegrated relative measurements
preint_dp = []   # delta position
preint_dv = []   # delta velocity
preint_dth = []  # delta heading

for kf in range(n_keyframes - 1):
    start = kf * kf_interval
    end   = (kf + 1) * kf_interval
    
    # Preintegrate relative motion (in body frame of keyframe start)
    dp = np.zeros(2)  # relative position
    dv = np.zeros(2)  # relative velocity
    dtheta = 0.0      # relative heading change
    
    for k in range(start, min(end, N_pi - 1)):
        # Subtract estimated bias
        ax_corrected = pi_imu_ax[k] - preint_a_bias[0]
        ay_corrected = pi_imu_ay[k] - preint_a_bias[1]
        w_corrected  = pi_imu_w[k]  - preint_g_bias
        
        # Rotate to the frame of the start keyframe
        c, s = np.cos(dtheta), np.sin(dtheta)
        aw_x = c * ax_corrected - s * ay_corrected
        aw_y = s * ax_corrected + c * ay_corrected
        
        dp += dv * preint_dt + 0.5 * np.array([aw_x, aw_y]) * preint_dt**2
        dv += np.array([aw_x, aw_y]) * preint_dt
        dtheta += w_corrected * preint_dt
    
    preint_dp.append(dp)
    preint_dv.append(dv)
    preint_dth.append(dtheta)

preint_dp = np.array(preint_dp)
preint_dv = np.array(preint_dv)
preint_dth = np.array(preint_dth)

# Ground truth relative motions
gt_dp = []
gt_dth_list = []
for kf in range(n_keyframes - 1):
    i_start = kf * kf_interval
    i_end   = (kf + 1) * kf_interval
    # Ground truth relative position (in world frame)
    dx = px[i_end] - px[i_start]
    dy = py[i_end] - py[i_start]
    # Rotate to body frame of start
    c, s = np.cos(pth[i_start]), np.sin(pth[i_start])
    gt_dp.append([c * dx + s * dy, -s * dx + c * dy])
    gt_dth_list.append(pth[i_end] - pth[i_start])

gt_dp = np.array(gt_dp)
gt_dth_arr = np.array(gt_dth_list)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

kf_times = np.arange(n_keyframes - 1) / keyframe_rate

axes[0].plot(kf_times, gt_dp[:, 0], 'k', lw=2, label='True Δx (body)')
axes[0].plot(kf_times, preint_dp[:, 0], 'steelblue', lw=1.5, ls='--', label='Preintegrated Δx')
axes[0].set_xlabel('Keyframe time (s)'); axes[0].set_ylabel('Δposition (m)')
axes[0].set_title('Preintegrated Δx'); axes[0].legend(fontsize=9)

axes[1].plot(kf_times, gt_dp[:, 1], 'k', lw=2, label='True Δy (body)')
axes[1].plot(kf_times, preint_dp[:, 1], 'tomato', lw=1.5, ls='--', label='Preintegrated Δy')
axes[1].set_xlabel('Keyframe time (s)'); axes[1].set_ylabel('Δposition (m)')
axes[1].set_title('Preintegrated Δy'); axes[1].legend(fontsize=9)

axes[2].plot(kf_times, np.degrees(gt_dth_arr), 'k', lw=2, label='True Δθ')
axes[2].plot(kf_times, np.degrees(preint_dth), 'forestgreen', lw=1.5, ls='--', label='Preintegrated Δθ')
axes[2].set_xlabel('Keyframe time (s)'); axes[2].set_ylabel('Δheading (deg)')
axes[2].set_title('Preintegrated Δθ'); axes[2].legend(fontsize=9)

plt.tight_layout(); plt.show()

dp_err = np.sqrt(np.sum((preint_dp - gt_dp)**2, axis=1))
dth_err = np.abs(preint_dth - gt_dth_arr)
print(f'Mean preintegration error:')
print(f'  Δposition: {np.mean(dp_err):.4f} m')
print(f'  Δheading:  {np.degrees(np.mean(dth_err)):.4f} deg')

**What to observe:** The preintegrated relative motions closely match the ground truth. Small errors come from noise and residual bias (after subtracting the estimated bias). These preintegrated factors can be used in a graph optimization framework (Chapter 38) alongside visual factors, creating a tightly coupled VIO system.

---
## 40.4 VIO Initialization

Before a VIO system can run, it needs to estimate several quantities that are not directly observable from a single frame:

1. **Initial velocity**: the IMU cannot measure position or velocity directly; it measures acceleration.
2. **Gravity direction**: needed to separate gravity from motion acceleration.
3. **Accelerometer bias**: must be estimated for accurate integration.
4. **Scale** (monocular only): a single camera cannot determine absolute scale.

The typical initialization procedure:
1. Run **visual SfM** (structure from motion) on the first few frames to get relative poses and 3D points.
2. **Align** the visual poses with IMU data: fit the IMU preintegrated measurements to the visual trajectory.
3. From this alignment, recover velocity, gravity, and bias.

Below, we demonstrate a simplified initialization: given a short trajectory with both camera and IMU data, estimate the initial velocity and accelerometer bias.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
init_duration = 3.0       # initialization window (s)  try 1.0, 3.0, 5.0
init_cam_rate = 10        # camera rate during init (Hz)
init_imu_rate = 200       # IMU rate (Hz)
init_cam_noise = 0.03     # camera noise (m)
init_a_noise  = 0.04      # accel noise (m/s²)
init_g_noise  = 0.003     # gyro noise (rad/s)
init_true_bias = np.array([0.012, -0.007])   # true accel bias
init_true_gbias = 0.0003  # true gyro bias
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(88)
dt_init = 1.0 / init_imu_rate

# Generate a short trajectory for initialization
t_init, ix, iy, ivx, ivy, iax, iay, ith, iw = \
    generate_trajectory(init_duration * 3, dt_init, shape='loop')  # longer traj, use first portion
N_init = int(init_duration / dt_init)
t_init = t_init[:N_init]; ix = ix[:N_init]; iy = iy[:N_init]
ivx = ivx[:N_init]; ivy = ivy[:N_init]; iax = iax[:N_init]; iay = iay[:N_init]
ith = ith[:N_init]; iw = iw[:N_init]

# Simulate sensors
init_imu_ax, init_imu_ay, init_imu_w = simulate_imu(
    iax, iay, ith, iw, dt_init, init_a_noise, init_g_noise, init_true_bias, init_true_gbias)

init_ci, init_cx, init_cy = simulate_camera(
    ix, iy, init_cam_noise, init_cam_rate, init_imu_rate)
# Only keep camera observations within our init window
mask = init_ci < N_init
init_ci = init_ci[mask]; init_cx = init_cx[mask]; init_cy = init_cy[mask]

# === Initialization: Estimate velocity and bias ===
# Method: Least squares fit.
# Between consecutive camera frames, the position change is:
#   p_{j} - p_{i} = v_i * Δt + ∫∫ a dt dt
# With unknown initial velocity v0 and unknown bias b_a,
# we set up a linear system and solve.

n_cam = len(init_ci)
if n_cam >= 3:
    # Build least squares: for each camera pair (i, j),
    # p_cam[j] - p_cam[i] ~ v0 * (t_j - t_0) + integrated_accel[0..j] - v0 * (t_i - t_0) - integrated_accel[0..i]
    # We solve for v0 and bias correction.
    
    # First, integrate IMU from start with zero initial velocity and zero bias correction
    # Then the residual tells us what v0 and bias must be.
    
    # Compute integrated velocity and position (no bias subtraction, v0=0)
    int_vx = np.cumsum(init_imu_ax) * dt_init  # velocity from accel integration
    int_vy = np.cumsum(init_imu_ay) * dt_init
    int_px = np.cumsum(int_vx) * dt_init  # position from velocity integration
    int_py = np.cumsum(int_vy) * dt_init
    
    # At camera frame k at time t_k:
    #   cam_pos[k] ~ p0 + v0 * t_k + int_pos[k] - 0.5 * bias * t_k^2
    # Rearranging:
    #   cam_pos[k] - p0 - int_pos[k] = v0 * t_k - 0.5 * bias * t_k^2
    # This is linear in [v0x, v0y, bx, by]
    
    A = []
    b_vec = []
    p0 = np.array([init_cx[0], init_cy[0]])
    t0 = t_init[init_ci[0]]
    
    for k in range(1, n_cam):
        idx = init_ci[k]
        tk = t_init[idx] - t0
        
        # Residual
        res_x = init_cx[k] - p0[0] - int_px[idx]
        res_y = init_cy[k] - p0[1] - int_py[idx]
        
        # Row for x: v0x * tk - 0.5 * bx * tk^2 = res_x
        A.append([tk, 0, -0.5 * tk**2, 0])
        b_vec.append(res_x)
        # Row for y: v0y * tk - 0.5 * by * tk^2 = res_y
        A.append([0, tk, 0, -0.5 * tk**2])
        b_vec.append(res_y)
    
    A = np.array(A)
    b_vec = np.array(b_vec)
    
    # Solve least squares
    result, _, _, _ = np.linalg.lstsq(A, b_vec, rcond=None)
    est_v0 = result[:2]
    est_bias_init = result[2:4]
    
    true_v0 = np.array([ivx[init_ci[0]], ivy[init_ci[0]]])
    
    print('=== VIO Initialization Results ===')
    print(f'Initial velocity:')
    print(f'  True:      [{true_v0[0]:.4f}, {true_v0[1]:.4f}] m/s')
    print(f'  Estimated: [{est_v0[0]:.4f}, {est_v0[1]:.4f}] m/s')
    print(f'  Error:     {np.linalg.norm(est_v0 - true_v0):.4f} m/s')
    print(f'\nAccelerometer bias:')
    print(f'  True:      [{init_true_bias[0]:.4f}, {init_true_bias[1]:.4f}] m/s²')
    print(f'  Estimated: [{est_bias_init[0]:.4f}, {est_bias_init[1]:.4f}] m/s²')
    print(f'  Error:     {np.linalg.norm(est_bias_init - init_true_bias):.4f} m/s²')
else:
    print('Not enough camera frames for initialization. Increase init_duration.')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
# Visualize initialization quality vs window length
init_windows = [0.5, 1.0, 2.0, 3.0, 5.0]  # seconds
n_init_trials = 30
# ─────────────────────────────────────────────────────────────────────────────

v_errors = []
b_errors = []

for win in init_windows:
    v_errs_trial = []
    b_errs_trial = []
    
    for trial in range(n_init_trials):
        np.random.seed(1000 + trial)
        N_w = int(win / dt_init)
        if N_w > N_init:
            N_w = N_init
        
        # Simulate noisy sensors for this trial
        t_imu_ax, t_imu_ay, t_imu_w = simulate_imu(
            iax[:N_w], iay[:N_w], ith[:N_w], iw[:N_w], dt_init,
            init_a_noise, init_g_noise, init_true_bias, init_true_gbias)
        t_ci, t_cx, t_cy = simulate_camera(
            ix[:N_w], iy[:N_w], init_cam_noise, init_cam_rate, init_imu_rate)
        msk = t_ci < N_w
        t_ci = t_ci[msk]; t_cx = t_cx[msk]; t_cy = t_cy[msk]
        
        if len(t_ci) < 3:
            continue
        
        int_vx_t = np.cumsum(t_imu_ax) * dt_init
        int_vy_t = np.cumsum(t_imu_ay) * dt_init
        int_px_t = np.cumsum(int_vx_t) * dt_init
        int_py_t = np.cumsum(int_vy_t) * dt_init
        
        A_t = []; b_t = []
        p0_t = np.array([t_cx[0], t_cy[0]])
        t0_t = t_init[t_ci[0]] if t_ci[0] < len(t_init) else 0
        
        valid = True
        for k in range(1, len(t_ci)):
            idx_k = t_ci[k]
            if idx_k >= len(int_px_t):
                valid = False
                break
            tk = t_init[idx_k] - t0_t if idx_k < len(t_init) else k * (1.0/init_cam_rate)
            res_x = t_cx[k] - p0_t[0] - int_px_t[idx_k]
            res_y = t_cy[k] - p0_t[1] - int_py_t[idx_k]
            A_t.append([tk, 0, -0.5*tk**2, 0]); b_t.append(res_x)
            A_t.append([0, tk, 0, -0.5*tk**2]); b_t.append(res_y)
        
        if not valid or len(A_t) < 4:
            continue
        
        res_t, _, _, _ = np.linalg.lstsq(np.array(A_t), np.array(b_t), rcond=None)
        v_err = np.linalg.norm(res_t[:2] - np.array([ivx[t_ci[0]], ivy[t_ci[0]]]))
        b_err = np.linalg.norm(res_t[2:4] - init_true_bias)
        v_errs_trial.append(v_err)
        b_errs_trial.append(b_err)
    
    v_errors.append(np.mean(v_errs_trial) if v_errs_trial else np.nan)
    b_errors.append(np.mean(b_errs_trial) if b_errs_trial else np.nan)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(init_windows, v_errors, 'o-', color='steelblue', lw=2, ms=8)
axes[0].set_xlabel('Initialization window (s)'); axes[0].set_ylabel('Velocity error (m/s)')
axes[0].set_title('Initial Velocity Estimation Error')

axes[1].plot(init_windows, b_errors, 's-', color='tomato', lw=2, ms=8)
axes[1].set_xlabel('Initialization window (s)'); axes[1].set_ylabel('Bias error (m/s²)')
axes[1].set_title('Bias Estimation Error')

plt.tight_layout(); plt.show()

for w, ve, be in zip(init_windows, v_errors, b_errors):
    print(f'  Window {w:.1f}s: velocity err = {ve:.4f} m/s, bias err = {be:.4f} m/s²')

**Key insight:** Longer initialization windows give better estimates of both velocity and bias. In practice, VIO systems need 1 to 3 seconds of motion (not stationary!) to initialize. The motion provides the excitation needed to separate velocity from bias. A stationary initialization can estimate bias but not velocity.

---
## Capstone: Full VIO Pipeline

A robot drives a loop for 30 seconds. We simulate:
- IMU at 100 Hz with noise and bias
- Camera at 10 Hz with position noise

We implement:
1. IMU integration between camera frames (prediction)
2. Camera based pose correction (Kalman update)
3. Online bias estimation within the Kalman state

We compare: IMU only, camera only, and fused VIO.

In [ ]:
class VIOWithBiasEstimation:
    """VIO with online bias estimation.
    
    State: [x, y, vx, vy, theta, b_ax, b_ay, b_g] (8 dimensional)
    The filter estimates the IMU biases alongside the pose.
    """
    def __init__(self, x0, y0, vx0, vy0, theta0,
                 a_noise, g_noise, cam_noise, bias_drift=1e-5):
        self.state = np.array([x0, y0, vx0, vy0, theta0, 0.0, 0.0, 0.0])
        self.P = np.diag([0.01, 0.01, 0.1, 0.1, 0.01, 0.05, 0.05, 0.01])
        
        self.Q_a = a_noise**2
        self.Q_g = g_noise**2
        self.Q_bias = bias_drift  # bias random walk variance rate
        self.R = cam_noise**2 * np.eye(2)
        
        # Observation: [x, y]
        self.H = np.zeros((2, 8))
        self.H[0, 0] = 1.0
        self.H[1, 1] = 1.0
    
    def predict(self, ax_b, ay_b, omega, dt):
        x, y, vx, vy, theta, bax, bay, bg = self.state
        
        # Subtract estimated biases
        ax_corr = ax_b - bax
        ay_corr = ay_b - bay
        w_corr  = omega - bg
        
        c, s = np.cos(theta), np.sin(theta)
        aw_x = c * ax_corr - s * ay_corr
        aw_y = s * ax_corr + c * ay_corr
        
        self.state = np.array([
            x + vx * dt,
            y + vy * dt,
            vx + aw_x * dt,
            vy + aw_y * dt,
            theta + w_corr * dt,
            bax,  # biases evolve as random walk
            bay,
            bg
        ])
        
        # Jacobian
        F = np.eye(8)
        F[0, 2] = dt
        F[1, 3] = dt
        F[2, 4] = (-s * ax_corr - c * ay_corr) * dt
        F[3, 4] = ( c * ax_corr - s * ay_corr) * dt
        # Derivatives w.r.t. biases
        F[2, 5] = -c * dt;  F[2, 6] = s * dt   # dvx/dbax, dvx/dbay
        F[3, 5] = -s * dt;  F[3, 6] = -c * dt  # dvy/dbax, dvy/dbay
        F[4, 7] = -dt  # dtheta/dbg
        
        Q = np.diag([
            self.Q_a * dt**2 * 0.25,
            self.Q_a * dt**2 * 0.25,
            self.Q_a * dt,
            self.Q_a * dt,
            self.Q_g * dt,
            self.Q_bias * dt,
            self.Q_bias * dt,
            self.Q_bias * dt * 0.1
        ])
        
        self.P = F @ self.P @ F.T + Q
    
    def update(self, z_cam):
        innovation = z_cam - self.H @ self.state
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.state = self.state + K @ innovation
        self.P = (np.eye(8) - K @ self.H) @ self.P
        return innovation
    
    @property
    def position(self): return self.state[:2]
    @property
    def velocity(self): return self.state[2:4]
    @property
    def heading(self): return self.state[4]
    @property
    def accel_bias(self): return self.state[5:7]
    @property
    def gyro_bias(self): return self.state[7]

print('VIOWithBiasEstimation class defined.')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
cap_duration   = 30.0     # seconds
cap_imu_rate   = 100      # Hz  (lower for faster simulation)
cap_cam_rate   = 10       # Hz
cap_cam_noise  = 0.05     # m
cap_a_noise    = 0.05     # m/s²
cap_g_noise    = 0.004    # rad/s
cap_a_bias     = np.array([0.02, -0.01])   # true accel bias
cap_g_bias     = 0.0006   # true gyro bias
cap_bias_drift = 1e-5     # bias random walk rate in filter
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
cap_dt = 1.0 / cap_imu_rate
cap_cam_every = int(cap_imu_rate / cap_cam_rate)

# Ground truth
tc, cx_gt, cy_gt, cvx, cvy, cax, cay, cth, cw = \
    generate_trajectory(cap_duration, cap_dt, shape='loop')
N_c = len(tc)

# Simulate sensors
c_imu_ax, c_imu_ay, c_imu_w = simulate_imu(
    cax, cay, cth, cw, cap_dt, cap_a_noise, cap_g_noise, cap_a_bias, cap_g_bias)
c_cam_idx, c_cam_x, c_cam_y = simulate_camera(
    cx_gt, cy_gt, cap_cam_noise, cap_cam_rate, cap_imu_rate)

# === 1. IMU only ===
imu_ox, imu_oy, _, _, _ = integrate_imu_2d(
    c_imu_ax, c_imu_ay, c_imu_w, cap_dt,
    cx_gt[0], cy_gt[0], cvx[0], cvy[0], cth[0])

# === 2. Camera only (interpolated) ===
cam_ox = np.interp(np.arange(N_c), c_cam_idx, c_cam_x)
cam_oy = np.interp(np.arange(N_c), c_cam_idx, c_cam_y)

# === 3. Fused VIO with bias estimation ===
vio_cap = VIOWithBiasEstimation(
    cx_gt[0], cy_gt[0], cvx[0], cvy[0], cth[0],
    cap_a_noise, cap_g_noise, cap_cam_noise, cap_bias_drift)

vio_x = np.zeros(N_c); vio_y = np.zeros(N_c)
vio_x[0] = cx_gt[0]; vio_y[0] = cy_gt[0]
est_bax = np.zeros(N_c); est_bay = np.zeros(N_c)
est_bg = np.zeros(N_c)

cam_set_cap = set(c_cam_idx)

for i in range(1, N_c):
    vio_cap.predict(c_imu_ax[i-1], c_imu_ay[i-1], c_imu_w[i-1], cap_dt)
    
    if i in cam_set_cap:
        ci = np.searchsorted(c_cam_idx, i)
        if ci < len(c_cam_x):
            vio_cap.update(np.array([c_cam_x[ci], c_cam_y[ci]]))
    
    vio_x[i] = vio_cap.position[0]
    vio_y[i] = vio_cap.position[1]
    est_bax[i] = vio_cap.accel_bias[0]
    est_bay[i] = vio_cap.accel_bias[1]
    est_bg[i] = vio_cap.gyro_bias

# Errors
err_imu_cap  = np.sqrt((imu_ox - cx_gt)**2 + (imu_oy - cy_gt)**2)
err_cam_cap  = np.sqrt((cam_ox - cx_gt)**2 + (cam_oy - cy_gt)**2)
err_vio_cap  = np.sqrt((vio_x - cx_gt)**2 + (vio_y - cy_gt)**2)

print('Capstone simulation complete.')
print(f'Trajectory: {cap_duration:.0f}s, IMU at {cap_imu_rate} Hz, Camera at {cap_cam_rate} Hz')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
# (Visualization of capstone results)
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Trajectory comparison
ax = axes[0, 0]
ax.plot(cx_gt, cy_gt, 'k', lw=3, label='Ground truth', zorder=5)
ax.plot(imu_ox, imu_oy, 'tomato', lw=1, alpha=0.6, label='IMU only')
ax.plot(cam_ox, cam_oy, 'orange', lw=1, alpha=0.6, label='Camera only')
ax.plot(vio_x, vio_y, 'steelblue', lw=2, label='Fused VIO', zorder=4)
ax.plot(cx_gt[0], cy_gt[0], 'o', color='forestgreen', ms=10, zorder=6)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('Capstone: Trajectory Comparison'); ax.legend(fontsize=9)
ax.set_aspect('equal')

# Position error
ax = axes[0, 1]
ax.plot(tc, err_imu_cap, 'tomato', lw=1, alpha=0.7, label='IMU only')
ax.plot(tc, err_cam_cap, 'orange', lw=1, alpha=0.7, label='Camera only')
ax.plot(tc, err_vio_cap, 'steelblue', lw=1.5, label='Fused VIO')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Position error (m)')
ax.set_title('Position Error Over Time'); ax.legend(fontsize=9)

# Bias estimation
ax = axes[1, 0]
ax.plot(tc, est_bax, 'steelblue', lw=1.5, label=f'Est $b_{{ax}}$ (true={cap_a_bias[0]:.3f})')
ax.plot(tc, est_bay, 'tomato', lw=1.5, label=f'Est $b_{{ay}}$ (true={cap_a_bias[1]:.3f})')
ax.axhline(cap_a_bias[0], color='steelblue', ls='--', alpha=0.4)
ax.axhline(cap_a_bias[1], color='tomato', ls='--', alpha=0.4)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Estimated bias (m/s²)')
ax.set_title('Online Accelerometer Bias Estimation'); ax.legend(fontsize=9)

ax = axes[1, 1]
ax.plot(tc, np.degrees(est_bg) * 3600, 'forestgreen', lw=1.5,
        label=f'Est $b_g$ (true={np.degrees(cap_g_bias)*3600:.1f} deg/hr)')
ax.axhline(np.degrees(cap_g_bias) * 3600, color='k', ls='--', alpha=0.4, label='True bias')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Estimated bias (deg/hr)')
ax.set_title('Online Gyroscope Bias Estimation'); ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
# (Statistics summary)
# ─────────────────────────────────────────────────────────────────────────────

print('================================================================')
print('           CAPSTONE: VIO PERFORMANCE SUMMARY                    ')
print('================================================================')
print(f'  Trajectory: {cap_duration:.0f}s loop, IMU {cap_imu_rate}Hz, Camera {cap_cam_rate}Hz')
print('================================================================')
print(f'  IMU only:')
print(f'    Mean error:  {np.mean(err_imu_cap):8.3f} m')
print(f'    Max error:   {np.max(err_imu_cap):8.3f} m')
print(f'    Final error: {err_imu_cap[-1]:8.3f} m')
print(f'  Camera only:')
print(f'    Mean error:  {np.mean(err_cam_cap):8.3f} m')
print(f'    Max error:   {np.max(err_cam_cap):8.3f} m')
print(f'  Fused VIO:')
print(f'    Mean error:  {np.mean(err_vio_cap):8.3f} m')
print(f'    Max error:   {np.max(err_vio_cap):8.3f} m')
print(f'    Final error: {err_vio_cap[-1]:8.3f} m')
print('================================================================')
print(f'  Improvement over IMU:    {np.mean(err_imu_cap)/np.mean(err_vio_cap):6.1f}x')
print(f'  Improvement over Camera: {np.mean(err_cam_cap)/np.mean(err_vio_cap):6.1f}x')
print('================================================================')
print(f'  Final bias estimates:')
print(f'    Accel X: true={cap_a_bias[0]:+.4f}, est={est_bax[-1]:+.4f} m/s²')
print(f'    Accel Y: true={cap_a_bias[1]:+.4f}, est={est_bay[-1]:+.4f} m/s²')
print(f'    Gyro:    true={cap_g_bias:.5f}, est={est_bg[-1]:.5f} rad/s')
print('================================================================')

**Capstone takeaways:**

1. **Fusion is strictly better** than either sensor alone. The fused VIO estimate is both smooth (high rate from IMU) and drift free (anchored by camera).
2. **Online bias estimation** allows the filter to learn and compensate for IMU biases over time. The bias estimates converge toward the true values as more camera corrections provide observability.
3. **The camera makes biases observable.** Without camera corrections, the biases are unobservable (they look the same as actual motion). With camera corrections, the filter can distinguish bias from real acceleration.
4. This is the core principle behind every VIO system: ARKit, ARCore, VINS-Mono, OKVIS, MSCKF.

---
## Exercises

### Exercise 40.1: Complementary Filter for Orientation

Implement a **complementary filter** to estimate orientation from accelerometer and gyroscope. The gyroscope gives angular velocity (integrate for heading, but drifts). The accelerometer gives the gravity direction (stable long term, but noisy short term). The complementary filter blends them:

$$\hat{\theta}_k = \alpha \cdot (\hat{\theta}_{k-1} + \omega_k \Delta t) + (1 - \alpha) \cdot \theta_{\text{accel},k}$$

where $\alpha \in [0.95, 0.99]$ weights the gyroscope (fast, drifty) vs the accelerometer (slow, stable). Simulate a 3D IMU with pitch and roll. Show that the complementary filter tracks orientation better than either sensor alone. Try different values of $\alpha$.

In [ ]:
# Your code here

### Exercise 40.2: Camera Dropout

Real cameras fail sometimes (motion blur, featureless walls, darkness). Modify the capstone VIO pipeline to simulate camera dropout: between $t=10$ s and $t=15$ s, no camera observations are available. How does the VIO estimator behave during the dropout? How quickly does it recover when camera returns? Plot the error before, during, and after dropout.

In [ ]:
# Your code here

### Exercise 40.3: Observability Analysis

When the robot moves in a straight line at constant velocity, the accelerometer bias is **unobservable** (it looks the same as a change in velocity). When the robot accelerates or turns, the bias becomes observable. Demonstrate this by running the VIO filter on two trajectories: (a) a straight line and (b) a curvy path. Compare the bias estimation convergence for both. Which converges faster and why?

In [ ]:
# Your code here

### Exercise 40.4: Scale Recovery

A monocular camera cannot determine absolute scale; its visual odometry output is "up to scale." The IMU *can* determine scale because it measures acceleration in m/s$^2$. Simulate monocular visual odometry that reports position with an unknown scale factor $s$: $p_{\text{cam}} = s \cdot p_{\text{true}} + \text{noise}$. Set $s = 0.7$ (the camera thinks the world is 30% smaller). Show that fusing with IMU data recovers the correct scale. Hint: add a scale state to the Kalman filter.

In [ ]:
# Your code here

### Exercise 40.5: Multi-Rate Sensor Fusion

Extend the VIO pipeline to include a third sensor: a **GPS** that provides position at 1 Hz with 2 m accuracy. Implement the three way fusion: IMU (100 Hz), camera (10 Hz), GPS (1 Hz). The GPS is much noisier than the camera but provides absolute position (no drift at all). Show the trajectory estimate with and without GPS. When does GPS help the most? (Hint: it helps when the camera fails or accumulates drift.)

In [ ]:
# Your code here